In [4]:
# ================================================================
# WEEKLY BYM + FACTOR (Double PolyGamma)
# p10 version
# 5 parallel chains
# ================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import coo_matrix, csr_matrix, diags, bmat, block_diag
from scipy.sparse.csgraph import connected_components
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr
import pickle
from joblib import Parallel, delayed
from pathlib import Path

BASE_DIR = Path(r"D:\77\Research\temp\snow")

# ================================================================
# SETTINGS
# ================================================================

burn = 10000
thin = 2
tot_save = 5000
total_iters = burn + thin * tot_save
n_chains = 5
period = 52

# ================================================================
# 1 Load data
# ================================================================

no_nbs = np.array([
57,170,236,269,343,685,946,947,989,
1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

# ================================================================
# 2 Two largest components
# ================================================================

gdf = gpd.GeoDataFrame(
geometry=gpd.points_from_xy(coords_full[:,0], coords_full[:,1]),
crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6

W_full = (squareform(pdist(xy)) <= 0.22).astype(int)
np.fill_diagonal(W_full, 0)
W_full = csr_matrix(W_full)

n_comp, labels = connected_components(W_full, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
np.where(labels == order[0])[0],
np.where(labels == order[1])[0]
]))

coords = coords_full[keep]
y = y_full[keep]
W = W_full[keep][:, keep]

S, TT = y.shape
print("Using S =", S)

# ================================================================
# 3 CAR precision
# ================================================================

deg = np.array(W.sum(axis=1)).flatten()
Q_car = diags(deg) - W
I_S = diags(np.ones(S))

# ================================================================
# 4 Time scaling
# ================================================================

t_full = np.arange(1, TT+1)
t_scaled_full = (t_full - t_full.mean()) / t_full.std()

# ================================================================
# 5 p10 dataset
# ================================================================

loc_mask = (y[:, :-1] == 1)
row_idx, time_idx = np.where(loc_mask)

N = len(row_idx)

next_y = y[row_idx, time_idx+1]

kappa = (1 - next_y) - 0.5

t_raw = time_idx + 1
t_scaled = t_scaled_full[time_idx]

week_idx = (t_raw - 1) % 52

# ================================================================
# 6 Covariates
# ================================================================

cov4 = np.column_stack([
np.ones(N),
np.cos(2*np.pi*t_raw/period),
np.sin(2*np.pi*t_raw/period),
t_scaled
])

K_base = 4
K_total = 8

# ================================================================
# 7 Factor covariates
# ================================================================

lat_raw = coords[:,1]
lat = (lat_raw - lat_raw.mean()) / lat_raw.std()

elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
elev = (elev_raw[keep] - elev_raw[keep].mean()) / elev_raw[keep].std()

snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)

temp_full = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()
temp_full = temp_full[keep]
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

t_lat = t_scaled * lat[row_idx]
t_elev = t_scaled * elev[row_idx]
t_temp = t_scaled * temp_scaled[row_idx, time_idx]

# ================================================================
# 8 X_eta_base
# ================================================================

eta_dim = K_total*S + 3

rows_e, cols_e, vals_e = [], [], []

for i in range(N):

    s = row_idx[i]

    for k in range(K_base):

        xval = cov4[i,k]

        rows_e += [i,i]
        cols_e += [(2*k)*S+s, (2*k+1)*S+s]
        vals_e += [xval,xval]

    rows_e += [i,i,i]
    cols_e += [K_total*S, K_total*S+1, K_total*S+2]
    vals_e += [t_lat[i],t_elev[i],t_temp[i]]

X_eta_base = coo_matrix(
(vals_e,(rows_e,cols_e)),
shape=(N,eta_dim)
).tocsr()

# ================================================================
# 9 X_tau_base
# ================================================================

tau_dim = K_total * 52

rows_t, cols_t, vals_t = [], [], []

for i in range(N):

    w = week_idx[i]

    for k in range(K_base):

        xval = cov4[i,k]

        rows_t += [i,i]
        cols_t += [(2*k)*52+w, (2*k+1)*52+w]
        vals_t += [xval,xval]

X_tau_base = coo_matrix(
(vals_t,(rows_t,cols_t)),
shape=(N,tau_dim)
).tocsr()

# ================================================================
# 10 Priors
# ================================================================

Q_blocks = []

for k in range(K_base):

    Q_blocks.append(Q_car)
    Q_blocks.append(I_S)

Q_spatial = bmat(
[[Q_blocks[i] if i==j else None
for j in range(K_total)]
for i in range(K_total)],
format="csr"
)

Q_factor = diags(np.ones(3))
Q_eta = block_diag((Q_spatial, Q_factor), format="csr")

tau_prior_prec = diags(np.ones(tau_dim)/9)

# ================================================================
# 11 MCMC chain
# ================================================================

def run_chain(chain_id):

    np.random.seed(1000 + chain_id)

    curr_eta = np.zeros(eta_dim)
    curr_tau = np.ones(tau_dim)

    all_eta = np.zeros((eta_dim, tot_save), dtype=np.float32)
    all_tau = np.zeros((tau_dim, tot_save), dtype=np.float32)

    save_idx = 0

    for it in tqdm(range(total_iters), desc=f"chain {chain_id}"):
        if it % 200 == 0:
            print(f"chain {chain_id} iter {it}/{total_iters}")
        X_tilde_eta = X_eta_base.copy()

        rows, cols = X_tilde_eta.nonzero()

        sp_mask = cols < K_total*S
        rows_sp = rows[sp_mask]
        cols_sp = cols[sp_mask]

        j = cols_sp // S
        w = week_idx[rows_sp]

        tau_indices = j*52 + w

        X_tilde_eta.data[sp_mask] *= curr_tau[tau_indices]

        psi = X_tilde_eta @ curr_eta
        omega = random_polyagamma(1, psi)

        XtOmega = X_tilde_eta.T.multiply(omega)

        post_prec_eta = XtOmega @ X_tilde_eta + Q_eta
        post_prec_eta = (post_prec_eta + post_prec_eta.T)*0.5
        post_prec_eta = post_prec_eta.tocsc()

        rhs_eta = X_tilde_eta.T @ kappa

        factor = cholesky(post_prec_eta, mode="simplicial")

        mu = factor.solve_A(rhs_eta)

        z = np.random.randn(eta_dim)
        z = z / np.sqrt(factor.D())
        z = factor.solve_Lt(z)
        z = factor.apply_Pt(z)

        curr_eta = mu + z

        X_tilde_tau = X_tau_base.copy()

        rows_nz, cols_nz = X_tilde_tau.nonzero()

        j = cols_nz // 52
        s = row_idx[rows_nz]

        eta_indices = j*S + s

        X_tilde_tau.data *= curr_eta[eta_indices]

        gamma = curr_eta[K_total*S:K_total*S+3]
        X_factor = X_eta_base[:, K_total*S:K_total*S+3]

        c = X_factor @ gamma

        residual = kappa - omega*c

        XtOmega = X_tilde_tau.T.multiply(omega)

        post_prec_tau = XtOmega @ X_tilde_tau + tau_prior_prec
        post_prec_tau = (post_prec_tau + post_prec_tau.T)*0.5
        post_prec_tau = post_prec_tau.tocsc()

        rhs_tau = X_tilde_tau.T @ residual

        factor = cholesky(post_prec_tau, mode="simplicial")

        mu = factor.solve_A(rhs_tau)

        z = np.random.randn(tau_dim)
        z = z / np.sqrt(factor.D())
        z = factor.solve_Lt(z)
        z = factor.apply_Pt(z)

        curr_tau = mu + z

        if it >= burn and (it-burn)%thin==0:

            all_eta[:,save_idx] = curr_eta
            all_tau[:,save_idx] = curr_tau

            save_idx += 1

            if save_idx == tot_save:
                break

    with open(BASE_DIR / f"bym10_chain{chain_id}.pkl","wb") as f:

        pickle.dump({
            "eta":all_eta,
            "tau":all_tau
        },f)

    return chain_id

# ================================================================
# 12 RUN PARALLEL CHAINS
# ================================================================

Parallel(n_jobs=n_chains)(
delayed(run_chain)(i) for i in range(n_chains)
)

print("All chains finished")

Using S = 1557


KeyboardInterrupt: 

In [ ]:
# ================================================================
# WEEKLY BYM + FACTOR (Double PolyGamma)
# p01 version
# 5 parallel chains
# ================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import coo_matrix, csr_matrix, diags, bmat, block_diag
from scipy.sparse.csgraph import connected_components
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr
import pickle
from joblib import Parallel, delayed
from pathlib import Path
import os

BASE_DIR = Path(r"D:\77\Research\temp\snow")

burn = 10000
thin = 2
tot_save = 5000
total_iters = burn + thin * tot_save
n_chains = 5
period = 52

# ================================================================
# 1 Load data
# ================================================================

no_nbs = np.array([
57,170,236,269,343,685,946,947,989,
1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

print("Initial S0 =", y_full.shape[0])

# ================================================================
# 2 Two largest components
# ================================================================

gdf = gpd.GeoDataFrame(
geometry=gpd.points_from_xy(coords_full[:,0], coords_full[:,1]),
crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6

W_full = (squareform(pdist(xy)) <= 0.22).astype(int)
np.fill_diagonal(W_full, 0)
W_full = csr_matrix(W_full)

n_comp, labels = connected_components(W_full, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
np.where(labels == order[0])[0],
np.where(labels == order[1])[0]
]))

coords = coords_full[keep]
y = y_full[keep]
W = W_full[keep][:, keep]

S, TT = y.shape
print("Using S =", S)

# ================================================================
# CAR precision
# ================================================================

deg = np.array(W.sum(axis=1)).flatten()
Q_car = diags(deg) - W
I_S = diags(np.ones(S))

# ================================================================
# Time scaling
# ================================================================

t_full = np.arange(1, TT+1)
t_scaled_full = (t_full - t_full.mean()) / t_full.std()

# ================================================================
# p01 dataset
# ================================================================

loc_mask = (y[:, :-1] == 0)
row_idx, time_idx = np.where(loc_mask)

N = len(row_idx)

next_y = y[row_idx, time_idx+1]
kappa = next_y - 0.5

t_raw = time_idx + 1
t_scaled = t_scaled_full[time_idx]
week_idx = (t_raw - 1) % 52

# ================================================================
# covariates
# ================================================================

cov4 = np.column_stack([
np.ones(N),
np.cos(2*np.pi*t_raw/period),
np.sin(2*np.pi*t_raw/period),
t_scaled
])

K_base = 4
K_total = 8

# ================================================================
# Factor covariates
# ================================================================

lat_raw = coords[:,1]
lat = (lat_raw - lat_raw.mean()) / lat_raw.std()

elev_raw = pd.read_csv("curr_elev.csv").iloc[:,3].to_numpy()
elev = (elev_raw[keep] - elev_raw[keep].mean()) / elev_raw[keep].std()

snow_temp = pyreadr.read_r("snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)

temp_full = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()
temp_full = temp_full[keep]
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

t_lat = t_scaled * lat[row_idx]
t_elev = t_scaled * elev[row_idx]
t_temp = t_scaled * temp_scaled[row_idx, time_idx]

# ================================================================
# X_eta_base
# ================================================================

eta_dim = K_total*S + 3

rows_e, cols_e, vals_e = [], [], []

for i in range(N):

    s = row_idx[i]

    for k in range(K_base):

        xval = cov4[i,k]

        rows_e += [i,i]
        cols_e += [(2*k)*S+s, (2*k+1)*S+s]
        vals_e += [xval,xval]

    rows_e += [i,i,i]
    cols_e += [K_total*S, K_total*S+1, K_total*S+2]
    vals_e += [t_lat[i],t_elev[i],t_temp[i]]

X_eta_base = coo_matrix(
(vals_e,(rows_e,cols_e)),
shape=(N,eta_dim)
).tocsr()

# ================================================================
# X_tau_base
# ================================================================

tau_dim = K_total * 52

rows_t, cols_t, vals_t = [], [], []

for i in range(N):

    w = week_idx[i]

    for k in range(K_base):

        xval = cov4[i,k]

        rows_t += [i,i]
        cols_t += [(2*k)*52+w, (2*k+1)*52+w]
        vals_t += [xval,xval]

X_tau_base = coo_matrix(
(vals_t,(rows_t,cols_t)),
shape=(N,tau_dim)
).tocsr()

# ================================================================
# Priors
# ================================================================

Q_blocks = []

for k in range(K_base):

    Q_blocks.append(Q_car)
    Q_blocks.append(I_S)

Q_spatial = bmat(
[[Q_blocks[i] if i==j else None
for j in range(K_total)]
for i in range(K_total)],
format="csr"
)

Q_factor = diags(np.ones(3)/9)
Q_eta = block_diag((Q_spatial, Q_factor), format="csr")

tau_prior_prec = diags(np.ones(tau_dim)/9)

# ================================================================
# MCMC function
# ================================================================

def run_chain(chain_id):

    print(f"Starting chain {chain_id} | PID {os.getpid()}")

    np.random.seed(1000 + chain_id)

    curr_eta = np.zeros(eta_dim)
    curr_tau = np.ones(tau_dim)

    all_eta = np.zeros((eta_dim, tot_save), dtype=np.float32)
    all_tau = np.zeros((tau_dim, tot_save), dtype=np.float32)

    save_idx = 0

    for it in range(total_iters):

        if it % 1000 == 0:
            print(f"chain {chain_id}: iter {it}/{total_iters}")

        X_tilde_eta = X_eta_base.copy()

        rows, cols = X_tilde_eta.nonzero()

        sp_mask = cols < K_total*S
        rows_sp = rows[sp_mask]
        cols_sp = cols[sp_mask]

        j = cols_sp // S
        w = week_idx[rows_sp]

        tau_indices = j*52 + w

        X_tilde_eta.data[sp_mask] *= curr_tau[tau_indices]

        psi = X_tilde_eta @ curr_eta
        omega = random_polyagamma(1, psi)

        XtOmega = X_tilde_eta.T.multiply(omega)

        post_prec_eta = XtOmega @ X_tilde_eta + Q_eta
        post_prec_eta = (post_prec_eta + post_prec_eta.T)*0.5
        post_prec_eta = post_prec_eta.tocsc()

        rhs_eta = X_tilde_eta.T @ kappa

        factor = cholesky(post_prec_eta, mode="simplicial")
        mu = factor.solve_A(rhs_eta)

        z = np.random.randn(eta_dim)
        z = z / np.sqrt(factor.D())
        z = factor.solve_Lt(z)
        z = factor.apply_Pt(z)

        curr_eta = mu + z

        X_tilde_tau = X_tau_base.copy()

        rows_nz, cols_nz = X_tilde_tau.nonzero()

        j = cols_nz // 52
        s = row_idx[rows_nz]

        eta_indices = j*S + s

        X_tilde_tau.data *= curr_eta[eta_indices]

        gamma = curr_eta[K_total*S:K_total*S+3]
        X_factor = X_eta_base[:, K_total*S:K_total*S+3]

        c = X_factor @ gamma

        residual = kappa - omega*c

        XtOmega = X_tilde_tau.T.multiply(omega)

        post_prec_tau = XtOmega @ X_tilde_tau + tau_prior_prec
        post_prec_tau = (post_prec_tau + post_prec_tau.T)*0.5
        post_prec_tau = post_prec_tau.tocsc()

        rhs_tau = X_tilde_tau.T @ residual

        factor = cholesky(post_prec_tau, mode="simplicial")

        mu = factor.solve_A(rhs_tau)

        z = np.random.randn(tau_dim)
        z = z / np.sqrt(factor.D())
        z = factor.solve_Lt(z)
        z = factor.apply_Pt(z)

        curr_tau = mu + z

        if it >= burn and (it-burn)%thin==0:

            all_eta[:,save_idx] = curr_eta
            all_tau[:,save_idx] = curr_tau

            save_idx += 1

            if save_idx == tot_save:
                break

    with open(BASE_DIR / f"bym01_chain{chain_id}.pkl","wb") as f:

        pickle.dump({
            "eta":all_eta,
            "tau":all_tau
        },f)

    print(f"Chain {chain_id} finished")

    return chain_id

# ================================================================
# Run parallel chains
# ================================================================

Parallel(n_jobs=n_chains, backend="loky")(
delayed(run_chain)(i) for i in range(n_chains)
)

print("All chains finished")